# Rainfall Classification: Hyperparameter Tuning

This notebook tunes the LdaBoost-based classifier on the rain dataset with reproducible settings and concise reporting.


In [4]:
# Reproducible setup and imports
import os
import random
import logging
from typing import Dict

import numpy as np
import pandas as pd

import optuna
from optuna.samplers import TPESampler

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# Global seed
RANDOM_SEED: int = 42
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Import LdaBoost via relative path
import sys
REL_PROJECT_ROOT = "../../"
if REL_PROJECT_ROOT not in sys.path:
    sys.path.insert(0, REL_PROJECT_ROOT)
from LdaBoost import LdaBoost


In [ ]:
N_ESTIMATOR = 100
LEARNING_RATE = 0.1
MAX_DEPTH = 3

## Dataset and preprocessing


In [5]:
import re

def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [re.sub(r"[^A-Za-z0-9_]+", "_", c) for c in df.columns]
    return df

train_df = pd.read_csv("Data/train.csv", index_col="id")
train_df = clean_column_names(train_df)

y = train_df["rainfall"]
X = train_df.drop(columns="rainfall").copy()

le = LabelEncoder()
y_enc = le.fit_transform(y)

# train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=RANDOM_SEED, stratify=y_enc
)

features = list(X.columns)



## Optuna study for LdaBoost


In [6]:
def objective(trial: optuna.Trial) -> float:
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 800, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 6),
        "random_state": RANDOM_SEED,
    }

    # Use only training data for hyperparameter tuning
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    scores = []
    for tr_idx, te_idx in cv.split(X_train, y_train):
        X_tr, X_te = X_train.iloc[tr_idx], X_train.iloc[te_idx]
        y_tr, y_te = y_train[tr_idx], y_train[te_idx]

        model = LdaBoost(**params)
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)
        scores.append(accuracy_score(y_te, y_pred))
    return float(np.mean(scores))

sampler = TPESampler(seed=RANDOM_SEED)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=15)

best_params: Dict[str, object] = study.best_params
best_value = study.best_value
best_params, best_value


({'n_estimators': 500, 'learning_rate': 0.015958237752949748, 'max_depth': 2},
 0.863018315018315)

## Train final LdaBoost and test performance


In [10]:
# Train final model on full training set and evaluate on test set
final_model = LdaBoost(
    n_estimators=int(best_params.get("n_estimators", N_ESTIMATOR)),
    learning_rate=float(best_params.get("learning_rate", LEARNING_RATE)),
    max_depth=int(best_params.get("max_depth", MAX_DEPTH)),
    random_state=RANDOM_SEED,
)

# Train on full training set
final_model.fit(X_train, y_train)

# Evaluate on test set
y_pred = final_model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)
test_f1 = f1_score(y_test, y_pred, average="macro")

print(f"Final LdaBoost — Test Accuracy: {test_accuracy:.4f}")
print(f"Final LdaBoost — Test F1 macro: {test_f1:.4f}")


Final LdaBoost — Test Accuracy: 0.8584
Final LdaBoost — Test F1 macro: 0.8071
